PSD & FFT Plot

In [7]:
import os
import numpy as np
import matplotlib.pyplot as plt
from obspy import read, UTCDateTime
from scipy.signal import welch
from obspy.signal.util import next_pow_2

In [ ]:
# --- Mana Event Configuration ---
folder_path = r"E:/others/avalanches/mana/mana_avalanche/ori/raw"
output_dir = r"C:/work/all_paper/papers/p3/Final submission/sr/final_editted/plot1/a_psd"

# -------- Event Window --------
START_TIME = UTCDateTime("2025-02-27T23:45:00")
END_TIME   = UTCDateTime("2025-02-27T23:55:00")

# -------- Filter Parameters --------
freqmin = 1.0
freqmax = 5.0
corners = 4
os.makedirs(output_dir, exist_ok=True)


In [ ]:
# --- Manikaran Event Configuration ---
folder_path = r"E:/others/avalanches/manikaran/himachal_avalanche/raw/nearest"
output_dir = r"C:/work/all_paper/papers/p3/Final submission/sr/final_editted/plot_2/a_psd"

# -------- Event Window --------
START_TIME = UTCDateTime("2025-03-30T11:30:00")
END_TIME   = UTCDateTime("2025-03-30T11:35:00")

# -------- Filter Parameters --------
freqmin = 15.0
freqmax = 25.0
corners = 4
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# --- Plot PSD + FFT ---
def plot_psd_fft(trace, output_path):
    data = trace.data.astype(np.float64)
    fs = trace.stats.sampling_rate
    nfft = next_pow_2(len(data))

    # --- PSD using Welch ---
    freqs_psd, psd = welch(data, fs=fs, nperseg=fs*4)  # 4-sec windows
    psd_db = 10 * np.log10(psd + 1e-12)  # Avoid log(0)

    # --- FFT ---
    freqs_fft = np.fft.rfftfreq(nfft, d=1/fs)
    fft_vals = np.fft.rfft(data, n=nfft)
    fft_amp = np.abs(fft_vals)

    
    fig, axs = plt.subplots(2, 1, figsize=(10, 8))

    # PSD
    axs[0].semilogx(freqs_psd, psd_db, color='blue')
    axs[0].set_title("Power Spectral Density (PSD)")
    axs[0].set_xlabel("Frequency [Hz]")
    axs[0].set_ylabel("Power [dB]")
    axs[0].grid(True, which='both', linestyle='--', linewidth=0.5)

    # FFT
    axs[1].plot(freqs_fft, fft_amp, color='green')
    axs[1].set_title("FFT Amplitude Spectrum")
    axs[1].set_xlabel("Frequency [Hz]")
    axs[1].set_ylabel("Amplitude")
    axs[1].grid(True)

    plt.suptitle(f"Spectral Analysis: {trace.id} | {trace.stats.starttime.isoformat()} — {trace.stats.endtime.isoformat()}")
    plt.tight_layout()
    plt.savefig(output_path, dpi= 450)
    plt.close()


In [ ]:
# --- Loop Through Files ---
for file_name in os.listdir(folder_path):
    if not file_name.endswith(".mseed"):
        continue

    file_path = os.path.join(folder_path, file_name)
    try:
        stream = read(file_path)
        for tr in stream:

            tr = tr.copy()

            tr.trim(
                starttime=START_TIME,
                endtime=END_TIME,
                pad=False
            )
            if tr.stats.npts == 0:
                continue

            tr.detrend("demean")
            tr.detrend("linear")
            tr.taper(max_percentage=0.05)

            tr.filter(
                "bandpass",
                freqmin=freqmin,
                freqmax=freqmax,
                corners=corners,
                zerophase=True
            )

            times = tr.times("utcdatetime")
            #times = [datetime.utcfromtimestamp(t.timestamp) for t in times]
            trace_id = tr.id.replace(".", "_")
            out_file = os.path.join(output_dir, f"{trace_id}_spectra.png")
            print(f"Plotting PSD & FFT for {tr.id}")
            plot_psd_fft(tr, out_file)
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

print("All spectral plots saved.")

Plotting PSD & FFT for .SANK..BHN
Plotting PSD & FFT for .SANK..BHZ
Plotting PSD & FFT for .SANK..BHE
Plotting PSD & FFT for .RAJG..BHZ
Plotting PSD & FFT for .BARS..BHE
Plotting PSD & FFT for .BARS..BHN
Plotting PSD & FFT for .BARS..BHZ
Plotting PSD & FFT for .CHAK..BHE
Plotting PSD & FFT for .CHAK..BHN
Plotting PSD & FFT for .CHAK..BHZ
Plotting PSD & FFT for .GIYA..BHE
Plotting PSD & FFT for .GIYA..BHN
Plotting PSD & FFT for .GIYA..BHZ
Plotting PSD & FFT for .RAJG..BHE
Plotting PSD & FFT for .RAJG..BHN
All spectral plots saved.
